# 01 — Configuration and Data Loading

Load paths and analysis parameters from `config.yaml` (formal) or `config.example.yaml` (sample), then read the parcel time series matrix `Y` used for GLHMM training.

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = Path('..').resolve()
CONFIG_NAME = os.environ.get('BRAIN_STATES_CONFIG', 'config.yaml')
CONFIG_PATH = PROJECT_ROOT / CONFIG_NAME
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f'{CONFIG_NAME} not found. Use config.example.yaml (sample) or config.yaml (formal).'
    )

with open(CONFIG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

use_example_data = bool(cfg.get('use_example_data', False))
paths = cfg['paths']
analysis = cfg['analysis']

n_parcels = int(analysis['n_parcels'])
K = int(analysis['K'])
method = analysis['method']
n_timepoints = int(analysis['n_timepoints'])
n_subjects = int(analysis['n_subjects'])
random_seed = int(analysis.get('random_seed', 123))

DATA_DIR = (PROJECT_ROOT / paths['data_dir']).resolve()
HMM_DIR = (PROJECT_ROOT / paths['hmm_models_dir']).resolve()
RESULTS_DIR = (PROJECT_ROOT / paths['results_dir']).resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
HMM_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
print(f'  use_example_data={use_example_data}')
print(f'  n_parcels={n_parcels}, K={K}, method={method}')
print(f'  data_dir={DATA_DIR}')
print(f'  hmm_models_dir={HMM_DIR}')


Configuration loaded.
  use_example_data=True
  n_parcels=18, K=3, method=example
  data_dir=D:\文件同步\文件\数据分析结果\data_code_figure\adaptive_reserve_moderate\brain_states\example_data
  hmm_models_dir=D:\文件同步\文件\数据分析结果\data_code_figure\adaptive_reserve_moderate\brain_states\results\example\hmm_models


In [2]:
def load_full_Y_and_indices():
    """Load concatenated parcel time series and subject segment indices.

    Returns
    -------
    Y : ndarray, shape (T, P)
        Observation matrix.
    indices : ndarray, shape (N, 2)
        Per-subject [start, end) row ranges into Y.
    subjects : list of str
        Subject identifiers (anonymized labels in example mode).
    """
    if use_example_data:
        Y = pd.read_csv(DATA_DIR / 'parcel_timeseries.csv').values
        idx_df = pd.read_csv(DATA_DIR / 'subject_indices.csv')
        indices = idx_df[['start_row', 'end_row']].to_numpy(dtype=int)
        subjects = idx_df['subject'].astype(str).tolist()
        return Y, indices, subjects

    signal_file = DATA_DIR / f'all_schaefer_parcel{n_parcels}_{method}.csv'
    indices_file = DATA_DIR / f'all_schaefer_indices_parcel{n_parcels}_{method}.csv'
    if not signal_file.exists():
        raise FileNotFoundError(
            f'Missing {signal_file}. Place study inputs under data/ or use config.example.yaml.'
        )
    if not indices_file.exists():
        raise FileNotFoundError(
            f'Missing {indices_file}. Place study inputs under data/ or use config.example.yaml.'
        )

    Y = pd.read_csv(signal_file).values
    indices = pd.read_csv(indices_file).values.astype(int)
    if indices.ndim != 2 or indices.shape[1] != 2:
        raise ValueError(f'Expected indices shape (N, 2); got {indices.shape}')
    subjects = [f'sub-{i+1:02d}' for i in range(indices.shape[0])]
    return Y, indices, subjects


Y, indices, subjects = load_full_Y_and_indices()
print(f'Y shape (time x parcels): {Y.shape}')
print(f'indices shape (subjects x 2): {indices.shape}')
print(f'n_subjects (from indices): {len(subjects)}')


Y shape (time x parcels): (360, 18)
indices shape (subjects x 2): (3, 2)
n_subjects (from indices): 3


In [ ]:
# Basic integrity checks
if not np.isfinite(Y).all():
    n_bad = int(np.size(Y) - np.isfinite(Y).sum())
    raise ValueError(f'Y contains {n_bad} non-finite values; clean inputs before training.')
if Y.shape[1] != n_parcels:
    print(
        f'Warning: config n_parcels={n_parcels} but Y has {Y.shape[1]} columns; '
        'using observed column count for metadata.'
    )
    n_parcels_obs = int(Y.shape[1])
else:
    n_parcels_obs = n_parcels

if indices.shape[0] != n_subjects:
    print(
        f'Warning: config n_subjects={n_subjects} but indices has {indices.shape[0]} rows; '
        'using observed subject count.'
    )
    n_subjects_obs = int(indices.shape[0])
else:
    n_subjects_obs = n_subjects

expected_T = int(indices[-1, 1] - indices[0, 0]) if indices.size else 0
if Y.shape[0] != expected_T and indices.size:
    # Allow non-contiguous layouts; still require last end == T if starts at 0
    if int(indices[0, 0]) == 0 and int(indices[-1, 1]) == Y.shape[0]:
        pass
    else:
        print(
            f'Note: Y rows={Y.shape[0]}, indices span [{indices[0, 0]}, {indices[-1, 1]}).'
        )

print(f'Finite check OK. Using n_parcels={n_parcels_obs}, n_subjects={n_subjects_obs}.')


Finite check OK. Using n_parcels=18, n_subjects=3.


In [4]:
# Persist shared arrays / metadata for notebook 02
np.save(RESULTS_DIR / 'Y.npy', Y)
np.save(RESULTS_DIR / 'indices.npy', indices)

meta = {
    'use_example_data': use_example_data,
    'n_parcels': n_parcels_obs,
    'K': K,
    'method': method,
    'n_timepoints': n_timepoints,
    'n_subjects': n_subjects_obs,
    'random_seed': random_seed,
    'model_beta': analysis.get('model_beta', 'no'),
    'covtype': analysis.get('covtype', 'full'),
    'model_mean': analysis.get('model_mean', 'state'),
    'skip_train_if_no_glhmm': bool(analysis.get('skip_train_if_no_glhmm', False)),
    'Y_shape': list(Y.shape),
    'indices_shape': list(indices.shape),
    'subjects': subjects,
    'config_name': CONFIG_NAME,
}
meta_path = RESULTS_DIR / 'analysis_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print(f'Saved metadata: {meta_path}')
print('Saved Y.npy and indices.npy')


Saved metadata: D:\文件同步\文件\数据分析结果\data_code_figure\adaptive_reserve_moderate\brain_states\results\example\analysis_metadata.json
Saved Y.npy and indices.npy
